In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import sys
from pathlib import Path

from scipy.optimize import minimize_scalar
from scipy.special import softmax
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss, brier_score_loss
from sklearn.preprocessing import label_binarize

sys.path.append(str(Path('..').resolve()))
from src.calibration import compute_ece, bootstrap_ece

Path('../results').mkdir(exist_ok=True)
Path('../figures').mkdir(exist_ok=True)

print("Imports OK")

Imports OK


In [3]:
# Load dataset
df = pd.read_csv('../data/processed/alerce_dataset.csv')

ALERCE_CLASSES = [
    'SNIa', 'SNIbc', 'SNII', 'SLSN', 'QSO',
    'AGN', 'Blazar', 'CV/Nova', 'YSO', 'LPV',
    'E', 'DSCT', 'RRL', 'CEP', 'Periodic-Other'
]
EVAL_CLASSES = ['SNIa', 'SNII', 'SNIbc', 'SLSN']

alerce_to_idx = {cls: i for i, cls in enumerate(ALERCE_CLASSES)}
eval_to_idx   = {cls: i for i, cls in enumerate(EVAL_CLASSES)}

# Exclude TDE — not in ALeRCE taxonomy
df_no_tde = df[df['alerce_class'] != 'TDE'].copy()
df_no_tde['alerce_idx'] = df_no_tde['alerce_class'].map(alerce_to_idx)
df_no_tde['eval_idx']   = df_no_tde['alerce_class'].map(eval_to_idx)

y_true_15   = df_no_tde['alerce_idx'].values    # indices into ALERCE_CLASSES (0-14)
y_true_4    = df_no_tde['eval_idx'].values       # indices into EVAL_CLASSES (0-3)
y_proba_15  = df_no_tde[ALERCE_CLASSES].values   # full 15-class probabilities

# Extract and renormalise 4-class probabilities
eval_indices = [alerce_to_idx[c] for c in EVAL_CLASSES]
y_proba_4 = y_proba_15[:, eval_indices]
y_proba_4 = y_proba_4 / y_proba_4.sum(axis=1, keepdims=True)

print(f"Dataset: {len(df_no_tde):,} objects (TDE excluded)")
print(f"\nClass distribution:")
for cls in EVAL_CLASSES:
    n = (df_no_tde['alerce_class'] == cls).sum()
    print(f"  {cls}: {n}")
print(f"\ny_true_15 shape:  {y_true_15.shape}")
print(f"y_true_4 shape:   {y_true_4.shape}")
print(f"y_proba_15 shape: {y_proba_15.shape}")
print(f"y_proba_4 shape:  {y_proba_4.shape}")

Dataset: 1,576 objects (TDE excluded)

Class distribution:
  SNIa: 757
  SNII: 560
  SNIbc: 204
  SLSN: 55

y_true_15 shape:  (1576,)
y_true_4 shape:   (1576,)
y_proba_15 shape: (1576, 15)
y_proba_4 shape:  (1576, 4)


In [4]:
def temp_scale_15(probs_15, T):
    """Apply temperature scaling to 15-class probabilities."""
    log_p = np.log(np.clip(probs_15, 1e-10, 1.0))
    return softmax(log_p / T, axis=1)

def get_4class(probs_15_scaled):
    """Extract and renormalise 4 evaluated classes from 15-class output."""
    p4 = probs_15_scaled[:, eval_indices]
    return p4 / p4.sum(axis=1, keepdims=True)

def ece_objective(T, y_true_15, probs_15):
    """ECE on 15-class predictions — used to fit T."""
    scaled = temp_scale_15(probs_15, T)
    return compute_ece(y_true_15, scaled)[0]

def multiclass_brier(y_true_4, probs_4):
    """Mean Brier score across 4 classes."""
    y_bin = label_binarize(y_true_4, classes=[0, 1, 2, 3])
    return np.mean([brier_score_loss(y_bin[:, k], probs_4[:, k])
                    for k in range(4)])

def find_optimal_T(y_true_15, probs_15):
    """Fit T by minimising ECE on calibration set."""
    res = minimize_scalar(
        lambda T: ece_objective(T, y_true_15, probs_15),
        bounds=(0.1, 10.0), method='bounded'
    )
    return res.x

def compute_all_metrics(y_true_15, y_true_4, probs_15):
    """Compute ECE, Brier, log-loss on given predictions."""
    probs_4 = get_4class(probs_15)
    ece, _  = compute_ece(y_true_15, probs_15)
    brier   = multiclass_brier(y_true_4, probs_4)
    ll      = log_loss(y_true_4, probs_4, labels=[0,1,2,3])
    return {'ece': ece, 'brier': brier, 'logloss': ll}

def compute_perclass_ece(y_true_15, probs_15):
    """Per-class ECE on 15-class predictions."""
    results = {}
    for cls in EVAL_CLASSES:
        idx_15 = alerce_to_idx[cls]
        mask   = (y_true_15 == idx_15)
        if mask.sum() < 5:
            results[cls] = np.nan
            continue
        ece, _ = compute_ece(y_true_15[mask], probs_15[mask])
        results[cls] = ece
    return results

print("All functions defined")
print(f"eval_indices: {eval_indices}")
print(f"  {[ALERCE_CLASSES[i] for i in eval_indices]}")

All functions defined
eval_indices: [0, 2, 1, 3]
  ['SNIa', 'SNII', 'SNIbc', 'SLSN']


In [5]:
# Baseline metrics on full dataset
baseline = compute_all_metrics(y_true_15, y_true_4, y_proba_15)
baseline_perclass = compute_perclass_ece(y_true_15, y_proba_15)

print("Baseline metrics (no temperature scaling):")
print(f"  ECE:      {baseline['ece']:.4f}")
print(f"  Brier:    {baseline['brier']:.4f}")
print(f"  Log-loss: {baseline['logloss']:.4f}")
print(f"\nPer-class ECE:")
for cls, ece in baseline_perclass.items():
    n = (y_true_15 == alerce_to_idx[cls]).sum()
    print(f"  {cls:<8} (n={n:3d}): {ece:.4f}")

Baseline metrics (no temperature scaling):
  ECE:      0.2587
  Brier:    0.1268
  Log-loss: 0.9377

Per-class ECE:
  SNIa     (n=757): 0.3757
  SNII     (n=560): 0.0990
  SNIbc    (n=204): 0.3604
  SLSN     (n= 55): 0.2250


In [6]:
# 5-fold stratified cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Storage for fold results
fold_T_values = []
fold_metrics_before = []
fold_metrics_after  = []
fold_perclass_before = {cls: [] for cls in EVAL_CLASSES}
fold_perclass_after  = {cls: [] for cls in EVAL_CLASSES}

# Combined test predictions across all folds
all_test_true_15  = []
all_test_true_4   = []
all_test_proba_15 = []
all_test_proba_15_scaled = []

print("Running 5-fold stratified cross-validation...\n")

for fold, (cal_idx, test_idx) in enumerate(skf.split(y_proba_15, y_true_4)):
    
    # Split
    y_cal_15   = y_true_15[cal_idx]
    y_cal_4    = y_true_4[cal_idx]
    p_cal_15   = y_proba_15[cal_idx]
    
    y_test_15  = y_true_15[test_idx]
    y_test_4   = y_true_4[test_idx]
    p_test_15  = y_proba_15[test_idx]
    
    # Fit T on calibration fold
    T_opt = find_optimal_T(y_cal_15, p_cal_15)
    fold_T_values.append(T_opt)
    
    # Apply to test fold
    p_test_scaled = temp_scale_15(p_test_15, T_opt)
    
    # Metrics on test fold
    m_before = compute_all_metrics(y_test_15, y_test_4, p_test_15)
    m_after  = compute_all_metrics(y_test_15, y_test_4, p_test_scaled)
    fold_metrics_before.append(m_before)
    fold_metrics_after.append(m_after)
    
    # Per-class ECE on test fold
    pc_before = compute_perclass_ece(y_test_15, p_test_15)
    pc_after  = compute_perclass_ece(y_test_15, p_test_scaled)
    for cls in EVAL_CLASSES:
        fold_perclass_before[cls].append(pc_before[cls])
        fold_perclass_after[cls].append(pc_after[cls])
    
    # Accumulate combined test set
    all_test_true_15.append(y_test_15)
    all_test_true_4.append(y_test_4)
    all_test_proba_15.append(p_test_15)
    all_test_proba_15_scaled.append(p_test_scaled)
    
    print(f"Fold {fold+1}: T={T_opt:.4f}  "
          f"ECE {m_before['ece']:.4f}→{m_after['ece']:.4f}  "
          f"Brier {m_before['brier']:.4f}→{m_after['brier']:.4f}")

print(f"\nT values: {[round(t,4) for t in fold_T_values]}")
print(f"T mean: {np.mean(fold_T_values):.4f}  std: {np.std(fold_T_values):.4f}")

Running 5-fold stratified cross-validation...

Fold 1: T=0.3759  ECE 0.2763→0.0548  Brier 0.1241→0.0982
Fold 2: T=0.3717  ECE 0.2485→0.0664  Brier 0.1257→0.1007
Fold 3: T=0.3524  ECE 0.2801→0.0625  Brier 0.1269→0.1042
Fold 4: T=0.3549  ECE 0.2460→0.0520  Brier 0.1303→0.1092
Fold 5: T=0.3481  ECE 0.2478→0.0602  Brier 0.1268→0.1026

T values: [np.float64(0.3759), np.float64(0.3717), np.float64(0.3524), np.float64(0.3549), np.float64(0.3481)]
T mean: 0.3606  std: 0.0111


In [7]:
# Combine all test fold predictions
y_combined_15     = np.concatenate(all_test_true_15)
y_combined_4      = np.concatenate(all_test_true_4)
p_combined_15     = np.concatenate(all_test_proba_15)
p_combined_scaled = np.concatenate(all_test_proba_15_scaled)

# Final metrics on combined test set
final_before = compute_all_metrics(y_combined_15, y_combined_4, p_combined_15)
final_after  = compute_all_metrics(y_combined_15, y_combined_4, p_combined_scaled)

# Bootstrap CIs on combined test set
_, lo_before, hi_before = bootstrap_ece(y_combined_15, p_combined_15)
_, lo_after,  hi_after  = bootstrap_ece(y_combined_15, p_combined_scaled)

print("="*55)
print("COMBINED TEST SET RESULTS (5-fold CV)")
print("="*55)
print(f"\n{'Metric':<12} {'Before':>10} {'After':>10} {'Change':>10}")
print("-"*45)
print(f"{'ECE':<12} {final_before['ece']:>10.4f} {final_after['ece']:>10.4f} "
      f"{final_after['ece']-final_before['ece']:>+10.4f}")
print(f"{'ECE 95% CI':<12} {'['+str(round(lo_before,4))+',':>10} "
      f"{'['+str(round(lo_after,4))+',':>10}")
print(f"{'':12} {str(round(hi_before,4))+']':>10} {str(round(hi_after,4))+']':>10}")
print(f"{'Brier':<12} {final_before['brier']:>10.4f} {final_after['brier']:>10.4f} "
      f"{final_after['brier']-final_before['brier']:>+10.4f}")
print(f"{'Log-loss':<12} {final_before['logloss']:>10.4f} {final_after['logloss']:>10.4f} "
      f"{final_after['logloss']-final_before['logloss']:>+10.4f}")

print(f"\nT statistics across folds:")
print(f"  Mean: {np.mean(fold_T_values):.4f}")
print(f"  Std:  {np.std(fold_T_values):.4f}")
print(f"  Range: [{min(fold_T_values):.4f}, {max(fold_T_values):.4f}]")
print(f"\nECE reduction: {(1 - final_after['ece']/final_before['ece'])*100:.1f}%")

COMBINED TEST SET RESULTS (5-fold CV)

Metric           Before      After     Change
---------------------------------------------
ECE              0.2587     0.0151    -0.2436
ECE 95% CI     [0.2369,   [0.0173,
                0.2795]     0.045]
Brier            0.1268     0.1030    -0.0238
Log-loss         0.9377     0.7870    -0.1507

T statistics across folds:
  Mean: 0.3606
  Std:  0.0111
  Range: [0.3481, 0.3759]

ECE reduction: 94.2%


In [8]:
# Per-class ECE on combined test set
pc_before_combined = compute_perclass_ece(y_combined_15, p_combined_15)
pc_after_combined  = compute_perclass_ece(y_combined_15, p_combined_scaled)

print("Per-class ECE (combined test set):")
print(f"\n{'Class':<8} {'N':>5} {'Before':>8} {'After':>8} {'Change':>8} {'Mean T±std':>12}")
print("-"*55)
for cls in EVAL_CLASSES:
    idx = alerce_to_idx[cls]
    n   = (y_combined_15 == idx).sum()
    eb  = pc_before_combined[cls]
    ea  = pc_after_combined[cls]
    # Per-class fold means
    fold_means_before = [v for v in fold_perclass_before[cls] if not np.isnan(v)]
    fold_means_after  = [v for v in fold_perclass_after[cls]  if not np.isnan(v)]
    print(f"{cls:<8} {n:>5} {eb:>8.4f} {ea:>8.4f} {ea-eb:>+8.4f}   "
          f"{np.mean(fold_means_after):.4f}±{np.std(fold_means_after):.4f}")

print(f"\nNote: positive change = worsening after temperature scaling")

Per-class ECE (combined test set):

Class        N   Before    After   Change   Mean T±std
-------------------------------------------------------
SNIa       757   0.3757   0.1135  -0.2622   0.1310±0.0042
SNII       560   0.0990   0.1880  +0.0890   0.2000±0.0259
SNIbc      204   0.3604   0.1087  -0.2517   0.1442±0.0270
SLSN        55   0.2250   0.1401  -0.0849   0.2115±0.0529

Note: positive change = worsening after temperature scaling


In [ ]:
# Class-specific temperature scaling
# Fit a separate T_k for each class on calibration folds

def temp_scale_classwise(probs_15, T_dict):
    """
    Apply class-specific temperature scaling.
    T_dict: {class_name: T_value}
    Each class's probability column is scaled by its own T,
    then the full vector is renormalised to sum to 1.
    """
    log_p = np.log(np.clip(probs_15, 1e-10, 1.0))
    scaled = log_p.copy()
    for cls, T in T_dict.items():
        idx = alerce_to_idx[cls]
        scaled[:, idx] = log_p[:, idx] / T
    return softmax(scaled, axis=1)

def find_classwise_T(y_true_15, probs_15):
    """Fit T_k per class by minimising per-class ECE on calibration set."""
    T_dict = {}
    for cls in EVAL_CLASSES:
        idx  = alerce_to_idx[cls]
        mask = (y_true_15 == idx)
        if mask.sum() < 10:
            T_dict[cls] = 1.0
            continue
        
        # Extract class subset once — no re-masking inside lambda
        y_cls = y_true_15[mask]
        p_cls = probs_15[mask]
        
        res = minimize_scalar(
            lambda T, y=y_cls, p=p_cls: compute_ece(
                y, temp_scale_classwise(p, {cls: T})
            )[0],
            bounds=(0.1, 10.0), method='bounded'
        )
        T_dict[cls] = res.x
    return T_dict

# 5-fold CV with class-specific T
fold_T_classwise = []
all_cw_scaled = []

print("5-fold CV with class-specific temperature scaling...\n")

for fold, (cal_idx, test_idx) in enumerate(skf.split(y_proba_15, y_true_4)):

    y_cal_15  = y_true_15[cal_idx]
    p_cal_15  = y_proba_15[cal_idx]
    y_test_15 = y_true_15[test_idx]
    y_test_4  = y_true_4[test_idx]
    p_test_15 = y_proba_15[test_idx]

    # Fit class-specific T on calibration fold
    T_dict = find_classwise_T(y_cal_15, p_cal_15)
    fold_T_classwise.append(T_dict)

    # Apply to test fold
    p_test_cw = temp_scale_classwise(p_test_15, T_dict)
    all_cw_scaled.append(p_test_cw)

    m_after = compute_all_metrics(y_test_15, y_test_4, p_test_cw)
    T_str = '  '.join([f"{cls}:{v:.3f}" for cls, v in T_dict.items()])
    print(f"Fold {fold+1}: ECE→{m_after['ece']:.4f}  Brier→{m_after['brier']:.4f}  |  {T_str}")

print(f"\nMean T per class across folds:")
for cls in EVAL_CLASSES:
    vals = [fd[cls] for fd in fold_T_classwise]
    print(f"  {cls:<8}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")